# Markerless Motion Analysis with Computer Vision
## A Yoga Pose Quality Assessment Prototype

This Jupyter notebook demonstrates an independent markerless motion
analysis pipeline designed to showcase modern computer vision and
software engineering practices.

The system analyzes uploaded or URL-based yoga videos using pretrained
models to extract 3D pose landmarks, compute biomechanical features,
and generate interpretable feedback.


In [1]:
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


ModuleNotFoundError: No module named 'google.protobuf'

ImportError: initialization failed

In [ ]:
from IPython.display import Video

VIDEO_PATH = r"C:\Users\pbhav\Downloads\yoga.mp4"  # or your actual filename, e.g. "yoga_pose.mp4"

Video(VIDEO_PATH)


In [ ]:
# Mediapipe Pose model initialization
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose_model = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.3
)


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

all_pose_landmarks = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Pose detection
    pose_results = pose_model.process(frame_rgb)
    if pose_results.pose_landmarks:
        pose_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in pose_results.pose_landmarks.landmark])
        all_pose_landmarks.append(pose_landmarks)
    else:
        all_pose_landmarks.append(np.zeros((33,3)))  # fill with zeros if pose not detected

cap.release()
pose_data = np.array(all_pose_landmarks)

print("Total frames:", frame_count)
print("Pose data shape:", pose_data.shape)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

all_landmarks = []
frame_count = 0
detected_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(frame_rgb)

    if results.pose_landmarks:
        detected_count += 1
        landmarks = np.array([
            [lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark
        ])
        all_landmarks.append(landmarks)

cap.release()

pose_data = np.array(all_landmarks)

print("Total frames:", frame_count)
print("Detected frames:", detected_count)
print("pose_data shape:", pose_data.shape)


In [ ]:
def normalize_pose(pose):
    left_hip = pose[23]
    right_hip = pose[24]
    hip_center = (left_hip + right_hip) / 2.0

    pose = pose - hip_center

    scale = np.linalg.norm(left_hip - right_hip)
    if scale < 1e-6:
        return None

    return pose / scale


normalized_pose = []

for p in pose_data:
    norm = normalize_pose(p)
    if norm is not None:
        normalized_pose.append(norm)

normalized_pose = np.array(normalized_pose)
print("Normalized frames:", len(normalized_pose))
print("Shape:", normalized_pose.shape)


In [ ]:
# Use first 30 frames as reference
reference_pose = np.mean(normalized_pose[:30], axis=0).flatten()

# Flatten each frame
user_embeddings = normalized_pose.reshape(normalized_pose.shape[0], -1)

# Cosine similarity per frame
from scipy.spatial.distance import cosine

similarity_scores = [
    1 - cosine(reference_pose, emb) for emb in user_embeddings
]

pose_quality_score = np.mean(similarity_scores) * 100
print("Pose Quality Score:", round(pose_quality_score, 2))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(similarity_scores)
plt.xlabel("Frame Index")
plt.ylabel("Similarity Score")
plt.title("Pose Consistency Over Time")
plt.show()


In [ ]:
if pose_quality_score > 80:
    feedback = "Good pose stability with minor alignment deviations."
elif pose_quality_score > 60:
    feedback = "Moderate inconsistencies detected. Focus on symmetry."
else:
    feedback = "Significant deviations detected. Consider adjusting posture."

print("Feedback:", feedback)
